In [1]:
import os
import torch
os.system(' pip uninstall --quiet numpy pyvista tetgen scipy -y')
os.system(' pip install  --upgrade --quiet pyvista tetgen scipy')
os.system(' pip install --upgrade --quiet numpy')
os.system(' pip install --upgrade --quiet numba scikit-learn scikit-image  ')
#os.system(' pip uninstall numpy -y')
#os.system(' pip install --quiet numpy==1.24.4')


0

In [2]:
import urllib.request
import pyvista as pv
import tetgen
import numpy as np
import torch
from scipy.sparse import coo_matrix
from pyspark.sql import SparkSession

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
from scipy.spatial import Delaunay


# Step 1: Download STL file
def download_stl(url, save_path):
    urllib.request.urlretrieve(url, save_path)
    print(f"Downloaded STL file to {save_path}")

# Example STL file URL (replace with your desired STL file)
stl_url = "https://people.sc.fsu.edu/~jburkardt/data/stla/cube.stl"
stl_file = "downloaded_model.stl"

sphere_url = "https://people.sc.fsu.edu/~jburkardt/data/stla/sphere.stl"
sphere_file = "sphere.stl"

#stl_url = sphere_url
#stl_file = sphere_file

download_stl(stl_url, stl_file)

# Step 2: Load STL file using PyVista
surface_mesh = pv.read(stl_file)

surface_mesh = surface_mesh.clean(tolerance=1e-5)
surface_mesh = surface_mesh.extract_surface()
surface_mesh = surface_mesh.triangulate()
surface_mesh = surface_mesh.connectivity(largest=True)
print("repaired mesh")
#surface_mesh = surface_mesh.triangulate()
surface_mesh = surface_mesh.subdivide(nsub=4, subfilter='linear')

os.system('pip install --quiet numpy')
# Step 3: Generate tetrahedral mesh using TetGen
tet = tetgen.TetGen(surface_mesh)
tet.tetrahedralize(order=1, mindihedral=20, minratio=2.0)
mesh = tet.grid

# Step 4: Extract cells and points
cells = mesh.cells_dict[10]  # VTK_TETRA = 10
points = mesh.points

spark = SparkSession.builder.appName("MeshFEM").getOrCreate()
sc = spark.sparkContext

tetra_data = []
for tet in cells:
    coords = [points[i].tolist() for i in tet]
    tetra_data.append((tet.tolist(), coords))

tetra_rdd = sc.parallelize(tetra_data)

def compute_fem_matrices(tet_data):
    #This function computes stiffness and mass matrices. It is run in parallel on pyspark
    from numpy import array, dot, linalg
    A_entries = []
    B_entries = []

    for tet_indices, tet_coords in [tet_data]:
        v0, v1, v2, v3 = array(tet_coords)
        J = np.column_stack((v1 - v0, v2 - v0, v3 - v0))
        det_J = np.linalg.det(J)
        if abs(det_J) < 1e-10:
            continue

        volume = abs(det_J) / 6.0
        inv_JT = np.linalg.inv(J).T
        grad_ref = array([[-1, -1, -1],
                          [1, 0, 0],
                          [0, 1, 0],
                          [0, 0, 1]])
        grads = inv_JT @ grad_ref.T
        grads = grads.T

        for i in range(4):
            for j in range(4):
                stiffness_entry = dot(grads[i], grads[j]) * volume
                A_entries.append((tet_indices[i], tet_indices[j], stiffness_entry))
                mass_entry = volume /10.0 if i == j else volume /20.0
                B_entries.append((tet_indices[i], tet_indices[j], mass_entry))

    return A_entries, B_entries
A_and_B_entries = tetra_rdd.map(compute_fem_matrices)
A_entries_rdd = A_and_B_entries.flatMap(lambda x: x[0])
B_entries_rdd = A_and_B_entries.flatMap(lambda x: x[1])

A_coo = A_entries_rdd.map(lambda x: ((x[0], x[1]), x[2])) \
                     .reduceByKey(lambda x, y: x+ y) \
                     .collect()
B_coo = B_entries_rdd.map(lambda x: ((x[0], x[1]), x[2])) \
                      .reduceByKey(lambda x, y: x + y) \
                      .collect()

def coo_list_to_matrix(coo_list, n):
    from scipy.sparse import coo_matrix
    rows, cols, data = zip(*[(i, j, v) for ((i, j), v) in coo_list])
    return coo_matrix((data, (rows, cols)), shape=(n, n))

n = points.shape[0]
A_global = coo_list_to_matrix(A_coo, n)
B_global = coo_list_to_matrix(B_coo, n)

# Step 1: Identify boundary nodes using PyVista
surf = mesh.extract_surface()
boundary_point_ids = np.unique(surf.point_data['vtkOriginalPointIds'])

# Step 2: Apply Dirichlet BCs to stiffness and mass matrices
from scipy.sparse import lil_matrix

A_mod = A_global.tolil()
B_mod = B_global.tolil()

for idx in boundary_point_ids:
    A_mod[idx, :] = 0
    A_mod[:, idx] = 0
    A_mod[idx, idx] = 1e16   # Enforces ϕ = 0 at this node

    B_mod[idx, :] = 0
    B_mod[:, idx] = 0
    B_mod[idx, idx] = 1.0   # Optional: Makes B non-singular

# Convert back to usable format
A_bc = A_mod.tocoo()
B_bc = B_mod.tocoo()




# --- Load matrices from COO data ---
# These should come from the PySpark aggregation step
# Assume A_global and B_global are scipy.sparse.coo_matrix

# Step 1: Convert to dense NumPy arrays
A_dense_np = A_bc.toarray()
B_dense_np = B_bc.toarray()

# Step 2: Convert to PyTorch tensors (CPU for now)
A = torch.tensor(A_dense_np, dtype=torch.float32)  # or float64 for precision
B = torch.tensor(B_dense_np, dtype=torch.float32)

# --- 💡 FUTURE: Enable this when you have GPU ---
# A = A.to("cuda")
# B = B.to("cuda")

# Step 3: Solve the generalized eigenvalue problem
# A x = λ B x
# → Convert to standard problem: B⁻¹ A x = λ x
B_inv = torch.linalg.inv(B)
eigen_matrix = B_inv @ A

# Solve (on CPU)
eigvals, eigvecs = torch.linalg.eig(eigen_matrix)

# Step 4: Extract real part (imaginary should be near-zero)
eigvals = eigvals.real
eigvecs = eigvecs.real

# --- Sort and interpret ---
sorted_indices = eigvals.argsort()
eigvals_sorted = eigvals[sorted_indices]
eigvecs_sorted = eigvecs[:, sorted_indices]

# Step 5: Inspect top modes
print("All eigenvalues (ω² or λ):")

for x in eigvals_sorted.numpy():
    print(x)

# --- 💡 FUTURE GPU Notes ---
# Once you get your RTX 3050 server:
# - Change all tensor initializations to `.to("cuda")`
# - Or run this as a remote API job on the GPU server

# Optional: Return eigenvectors if you want to plot the fields




Downloaded STL file to downloaded_model.stl
repaired mesh


/opt/conda/lib/python3.11/site-packages/pyvista/core/filters/data_set.py:1963: PyVistaDeprecationWarning: Use of `largest=True` is deprecated. Use 'largest' or `extraction_mode='largest'` instead.
  warnings.warn(
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/06/03 08:04:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


All eigenvalues (ω² or λ):
30.155382
61.507732
61.667614
61.76772
93.89347
93.96093
94.20228
116.65179
116.926414
117.467155
127.365875
150.1283
150.66731
151.03513
151.38295
151.86104
152.03609
185.23575
186.4045
186.84654
200.99863
201.10085
201.96375
211.17827
211.28221
213.26088
235.16737
235.99411
236.54433
237.45792
238.01614
239.3865
246.70601
247.59462
248.68552
272.6612
274.10733
274.99225
298.3692
301.23715
301.41147
301.9517
302.86722
305.52927
313.09973
317.5211
318.61813
321.39484
336.11432
337.56015
339.4366
341.15158
341.4371
342.68777
353.74622
355.28864
357.05338
358.0255
360.41562
362.99805
393.21564
395.13318
396.9179
398.30206
399.47205
402.18723
405.8565
408.4772
410.3278
421.7075
423.0508
428.4872
429.47495
431.5075
434.0317
435.66437
438.06766
442.905
461.89465
463.18396
466.0401
466.92844
469.72012
472.05704
474.60706
475.25095
482.61902
506.48206
508.34293
510.16714
512.61005
513.3556
515.0401
515.86035
518.4615
523.25525
524.2516
530.5247
532.4643
532.69855
53

In [3]:
import os
import pyvista as pv
import numpy as np

# Make sure your mesh and eigenvectors are ready
output_dir = "eigenmodes"
os.makedirs(output_dir, exist_ok=True)

# If needed: convert PyTorch tensor to numpy
eigvecs_np = eigvecs_sorted.cpu().numpy()

eigenmode_type = "cube"
# Export the first N modes
N = min(100,len(eigvecs_sorted))  # you can change this
for i in range(N):
    mode = eigvecs_np[:, i]
    mesh.point_data.clear()  # Clear previous fields
    mesh.point_data[f"mode_{i}"] = mode
    mesh.save(os.path.join(output_dir, f"{eigenmode_type}_eigenmode_{i:02d}.vtu"))

print(f"✅ Exported {N} eigenmodes to '{output_dir}/' for ParaView.")


✅ Exported 100 eigenmodes to 'eigenmodes/' for ParaView.


In [4]:
import urllib.request
import pyvista as pv

# Step 1: Download STL file
url = "https://people.sc.fsu.edu/~jburkardt/data/stla/sphere.stl"
filename = "sphere.stl"
urllib.request.urlretrieve(url, filename)
print(f"Downloaded STL to: {filename}")

# Step 2: Load STL
mesh = pv.read(filename)

# Step 3: Repair and clean the mesh
mesh = mesh.clean(tolerance=1e-5)
mesh = mesh.extract_surface()

# Step 4: Triangulate the mesh (critical!)
mesh = mesh.triangulate()

# Step 5: Keep only the largest connected surface
mesh = mesh.connectivity(largest=True)

# Step 6: Save the repaired, triangulated STL file
mesh.save("sphere_repaired_tri.stl")
print("Saved repaired triangulated mesh to: sphere_repaired_tri.stl")

# Optional: Visualize
# mesh.plot(show_edges=True)


Downloaded STL to: sphere.stl
Saved repaired triangulated mesh to: sphere_repaired_tri.stl
